In [ ]:
# our first code for putting together data for running regressions against ratings

from standard_imports import *

ratings = pd.read_parquet(data_path / 'S&P ratings.parquet')
financials = pd.read_parquet(data_path / 'compustat financials.parquet')
info = pd.read_parquet(data_path / 'company info.parquet')

# The first thing we do is to remove all companies without an industry indicator
# and any companies in the financial, real estate, and utility industries
ratings = pd.merge(ratings, info[['gvkey', 'gsector']], on = 'gvkey')
ratings = ratings[~ratings.gsector.isna()]
ratings = ratings[~ratings.gsector.isin(['40', '55', '60'])]

# identifying the first default for all defaulting comkpanies
defaults = ratings[ratings.rtg_symbol.isin(['D', 'SD', 'R'])].copy()
defaults = defaults[['gvkey', 'date']].groupby('gvkey', as_index = False).date.min()
defaults.columns = ['gvkey', 'default_date']

# we first pull in the financial data and filter it
financials = financials.rename({'at':'total_assets'}, axis = 1)
financials = financials[~pd.isnull(financials.total_assets)]
financials = financials[financials.total_assets > 0]

# making sure we restrict ourselves to the rated universe
# we don't know the default history of the non-rated universe
ratings = pd.read_parquet(data_path / 'S&P ratings.parquet')
financials = pd.merge(financials, ratings, on = 'gvkey')
financials = financials[(financials.data_date >= financials.date) & \
                        (financials.data_date < financials.end_date)]
financials = financials[~financials.rtg_symbol.isin(['NR','R','SD','D'])]
financials = financials.drop_duplicates()

# the following sets a default flag to 1 anytime there is a default occurring 6 to 18 months after the 
# statement date
financials = pd.merge(financials, defaults[['gvkey', 'default_date']], on = 'gvkey', how = 'left')
financials.loc[pd.isnull(financials.default_date), 'default_date'] = pd.Timestamp(2199,12,31)
financials['default_flag'] = 0

# I'm removing any records that are after about six months prior to a default
# Most analysts already know a company is in distress during this window.
# Also, most financials aren't released for six to ten weeks after the statement date
financials['default_minus_six_months'] = financials.default_date - DateOffset(months = 6)
financials = financials[financials.data_date < financials.default_minus_six_months]
del financials['default_minus_six_months']

financials['date_plus_six_months'] = financials.data_date + DateOffset(months = 6)
financials['date_plus_18_months'] = financials.data_date + DateOffset(months = 18)

financials['default_flag'] = 0
financials.loc[(financials.default_date > financials.date_plus_six_months) \
             & (financials.default_date <= financials.date_plus_18_months), 'default_flag'] = 1
financials = financials.drop_duplicates()

print(f'There are {len(financials):,} rows in data')
print(f'there are {financials.default_flag.sum()} defaults in the data')
